<a href="https://colab.research.google.com/github/gulshan0201/Deep-Learning/blob/main/LabExam2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. Build a GAN model to generate an image. Explain individual components present in generator and discriminator
2. Develop a RNN to predict the next word in a sequence. Explain the process with an example.
3. Explain the gate architecture of LSTM and develop an sequential model.
4. Design a detailed architecture of Attention model
5. Explain essential components of Transformers

#GAN
- if we print the image after running u will see the noise as image because epochs is 3 , it needed gpu so we run only till 3 epocs

In [ ]:
# Step 1: Import libraries
import tensorflow as tf
from tensorflow.keras import layers
import numpy as np

# Step 2: Load dataset (MNIST)
(x_train, _), _ = tf.keras.datasets.mnist.load_data()

# Normalize to [-1, 1]
x_train = (x_train - 127.5) / 127.5
x_train = x_train.reshape(-1, 784)

# Step 3: Build Generator
def build_generator():
    model = tf.keras.Sequential([
        layers.Dense(128, activation='relu', input_dim=100),
        layers.Dense(784, activation='tanh')
    ])
    return model

# Step 4: Build Discriminator
def build_discriminator():
    model = tf.keras.Sequential([
        layers.Dense(128, activation='relu', input_dim=784),
        layers.Dense(1, activation='sigmoid')
    ])
    return model

# Step 5: Create models
generator = build_generator()
discriminator = build_discriminator()

# Compile discriminator
discriminator.compile(optimizer='adam', loss='binary_crossentropy')

# Step 6: Combine GAN
discriminator.trainable = False

gan_input = layers.Input(shape=(100,))
fake_image = generator(gan_input)
gan_output = discriminator(fake_image)

gan = tf.keras.Model(gan_input, gan_output)
gan.compile(optimizer='adam', loss='binary_crossentropy')

# Step 7: Training loop
epochs = 3
batch_size = 64

for epoch in range(epochs):
    for i in range(0, x_train.shape[0], batch_size):
        # Real images
        real_images = x_train[i:i+batch_size]
        real_labels = np.ones((real_images.shape[0], 1))

        # Fake images
        noise = np.random.normal(0, 1, (real_images.shape[0], 100))
        fake_images = generator.predict(noise, verbose=0)
        fake_labels = np.zeros((real_images.shape[0], 1))

        # Train Discriminator
        d_loss_real = discriminator.train_on_batch(real_images, real_labels)
        d_loss_fake = discriminator.train_on_batch(fake_images, fake_labels)

        # Train Generator
        noise = np.random.normal(0, 1, (real_images.shape[0], 100))
        g_loss = gan.train_on_batch(noise, np.ones((real_images.shape[0], 1)))

    print(f"Epoch {epoch+1}, D Loss: {d_loss_real + d_loss_fake}, G Loss: {g_loss}")

Epoch 1, D Loss: 11.509380340576172, G Loss: 0.0018893451197072864
Epoch 2, D Loss: 12.273555755615234, G Loss: 0.0009493118850514293
Epoch 3, D Loss: 12.801660537719727, G Loss: 0.0006341212429106236


# RNN


In [ ]:
# Import libraries
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense
from tensorflow.keras.utils import to_categorical

# Dataset
data = """Machine learning is a field of artificial intelligence.
It allows systems to learn from data.
Deep learning is a subset of machine learning.
It uses neural networks.
Natural language processing helps computers understand human language.
RNN is used for sequential data like text and speech.
Word prediction is useful in chat applications and search engines."""

# Tokenization
tokenizer = Tokenizer()
tokenizer.fit_on_texts([data])
total_words = len(tokenizer.word_index) + 1

# Create sequences
input_sequences = []
for line in data.split('.'):
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        input_sequences.append(token_list[:i+1])

# Padding
max_len = max([len(seq) for seq in input_sequences])
input_sequences = pad_sequences(input_sequences, maxlen=max_len, padding='pre')

# Split input and output
X = input_sequences[:, :-1]
y = input_sequences[:, -1]

# One-hot encode output
y = to_categorical(y, num_classes=total_words)

# Model
model = Sequential()
model.add(Embedding(total_words, 10, input_length=max_len-1))
model.add(SimpleRNN(50))
model.add(Dense(total_words, activation='softmax'))

# Compile
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# Train
model.fit(X, y, epochs=100, verbose=0)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [ ]:
def predict_next_word(seed_text):
    token_list = tokenizer.texts_to_sequences([seed_text])[0]
    token_list = pad_sequences([token_list], maxlen=max_len-1, padding='pre')
    predicted = np.argmax(model.predict(token_list, verbose=0))

    for word, index in tokenizer.word_index.items():
        if index == predicted:
            return word
print(predict_next_word("machine learning"))

is


# LSTM

In [ ]:
# Import libraries
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.utils import to_categorical

# Dataset (simple paragraphs)
data = """Machine learning is a field of artificial intelligence.
It allows systems to learn from data.
Deep learning is a subset of machine learning.
It uses neural networks.
Natural language processing helps computers understand human language.
RNN is used for sequential data like text and speech.
Word prediction is useful in chat applications and search engines."""

# Tokenization
tokenizer = Tokenizer()
tokenizer.fit_on_texts([data])
total_words = len(tokenizer.word_index) + 1

# Create input sequences
input_sequences = []
for line in data.split('.'):
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        input_sequences.append(token_list[:i+1])

# Padding sequences
max_len = max(len(seq) for seq in input_sequences)
input_sequences = pad_sequences(input_sequences, maxlen=max_len, padding='pre')

# Split into input (X) and output (y)
X = input_sequences[:, :-1]
y = input_sequences[:, -1]

# One-hot encoding
y = to_categorical(y, num_classes=total_words)

# Build LSTM model
model = Sequential()
model.add(Embedding(total_words, 10, input_length=max_len-1))
model.add(LSTM(50))
model.add(Dense(total_words, activation='softmax'))

# Compile model
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# Train model
model.fit(X, y, epochs=100, verbose=0)

# Prediction function
def predict_next_word(seed_text):
    token_list = tokenizer.texts_to_sequences([seed_text])[0]
    token_list = pad_sequences([token_list], maxlen=max_len-1, padding='pre')

    predicted = np.argmax(model.predict(token_list, verbose=0))

    for word, index in tokenizer.word_index.items():
        if index == predicted:
            return word

# Example
print(predict_next_word("machine learning"))

is


# Attention

In [ ]:
# Import libraries
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.utils import to_categorical
import tensorflow as tf

# Dataset
data = ["deep learning models are powerful",
        "attention helps focus on important words",
        "neural networks process sequential data",
        "machine learning uses data for prediction"]

# Tokenization
tokenizer = Tokenizer()
tokenizer.fit_on_texts(data)
total_words = len(tokenizer.word_index) + 1

# Create sequences
input_sequences = []
for line in data:
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        input_sequences.append(token_list[:i+1])

# Padding
max_len = max(len(seq) for seq in input_sequences)
input_sequences = pad_sequences(input_sequences, maxlen=max_len, padding='pre')

X = input_sequences[:, :-1]
y = input_sequences[:, -1]
y = to_categorical(y, num_classes=total_words)

# -------- Simple Attention Layer --------
class SimpleAttention(tf.keras.layers.Layer):
    def call(self, inputs):
        score = tf.nn.softmax(inputs, axis=1)
        context = score * inputs
        context = tf.reduce_sum(context, axis=1)
        return context

# -------- Model --------
model = Sequential()
model.add(Embedding(total_words, 10, input_length=max_len-1))
model.add(LSTM(50, return_sequences=True))
model.add(SimpleAttention())   # ✅ Attention added here
model.add(Dense(total_words, activation='softmax'))

# Compile
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# Train
model.fit(X, y, epochs=100, verbose=0)

In [ ]:
def predict_next_word(seed_text):
    token_list = tokenizer.texts_to_sequences([seed_text])[0]
    token_list = pad_sequences([token_list], maxlen=max_len-1, padding='pre')
    predicted = np.argmax(model.predict(token_list, verbose=0))

    for word, index in tokenizer.word_index.items():
        if index == predicted:
            return word
print(predict_next_word("machine learning"))

is


# Transformers

In [ ]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Embedding, Dense, LayerNormalization, MultiHeadAttention, GlobalAveragePooling1D
from tensorflow.keras.models import Sequential
from tensorflow.keras.utils import to_categorical

# -------- Dataset --------
data = """machine learning is fun and powerful.
deep learning makes machines intelligent.
transformers are useful for language tasks."""

# -------- Tokenization --------
tokenizer = Tokenizer()
tokenizer.fit_on_texts([data])
total_words = len(tokenizer.word_index) + 1

# -------- Create Sequences --------
input_sequences = []
for line in data.split('.'):
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        input_sequences.append(token_list[:i+1])

# -------- Padding --------
max_len = max(len(seq) for seq in input_sequences)
input_sequences = pad_sequences(input_sequences, maxlen=max_len, padding='pre')

X = input_sequences[:, :-1]
y = input_sequences[:, -1]

# -------- Transformer Block --------
class TransformerBlock(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim):
        super().__init__()
        self.att = MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = tf.keras.Sequential([
            Dense(ff_dim, activation='relu'),
            Dense(embed_dim),
        ])
        self.norm1 = LayerNormalization()
        self.norm2 = LayerNormalization()

    def call(self, inputs):
        attn_output = self.att(inputs, inputs)
        out1 = self.norm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        return self.norm2(out1 + ffn_output)

# -------- Model --------
embed_dim = 32
num_heads = 2
ff_dim = 64

model = Sequential()
model.add(Embedding(total_words, embed_dim, input_length=max_len-1))
model.add(TransformerBlock(embed_dim, num_heads, ff_dim))
model.add(GlobalAveragePooling1D())
model.add(Dense(32, activation='relu'))
model.add(Dense(total_words, activation='softmax'))

# Compile
model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# Train
model.fit(X, y, epochs=100, verbose=0)

# -------- Prediction Function --------
def predict_next_word(text):
    token_list = tokenizer.texts_to_sequences([text])[0]
    token_list = pad_sequences([token_list], maxlen=max_len-1, padding='pre')

    predicted = np.argmax(model.predict(token_list, verbose=0))

    for word, index in tokenizer.word_index.items():
        if index == predicted:
            return word

# Example
print(predict_next_word("machine learning"))

is
